# All-Day Socket, Notebook 01: Virtual Patients

Notebook 01 builds the data for neural network to learn from and covers the following:

1. Loads the OpenLimbTT shape model
2. Splits a limb into skin and bones
3. Generates new realistic virtual limbs
4. Measures soft tissue thickness
5. Makes morning / midday / evening volume versions of each limb
6. Splits patients into train, validation, and test sets

## Install libraries and download the data


In [ ]:
!pip install -q trimesh
!git clone --depth 1 https://github.com/abel-research/OpenLimbTT.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.4/750.4 kB 8.6 MB/s eta 0:00:00
Cloning into 'OpenLimbTT'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 59 (delta 2), reused 49 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 53.00 MiB | 19.96 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [ ]:
# optional: if skipped data is stored in Colab's temporary storage instead of drive
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/all-day-socket/data'
except ImportError:          # not running in Colab
    OUT_DIR = 'data'
os.makedirs(OUT_DIR, exist_ok=True)
print('Saving outputs to:', OUT_DIR)

Mounted at /content/drive
Saving outputs to: /content/drive/MyDrive/all-day-socket/data


## Import tools and point to the model files


In [ ]:
import numpy as np
import trimesh
import pickle
import warnings
from scipy.spatial import cKDTree
import plotly.graph_objects as go

warnings.filterwarnings('ignore')   # hides a harmless scikit-learn version warning

MODEL_DIR = 'OpenLimbTT/version-2025-07'
print(sorted(os.listdir(MODEL_DIR)))

['Components16.npy', 'Generate Random Synthetic Model.py', 'LR.pkl', 'Mean-Bones-Full-Size.stl', 'Mean-Limb-Full-Size.stl', 'Mean-Limb.stl', 'Mean-Skin-Full-Size.stl', 'Mean-Skin.stl', 'Mode1-20th-Mode2-20th.stl', 'Mode1-20th-Mode2-80th.stl', 'Mode1-20th-Mode2-Mean.stl', 'Mode1-80th-Mode2-20th.stl', 'Mode1-80th-Mode2-80th.stl', 'Mode1-80th-Mode2-Mean.stl', 'Mode1-Mean-Mode2-20th.stl', 'Mode1-Mean-Mode2-80th.stl', 'version-notes']


## Load the average limb


In [ ]:
mean_limb = trimesh.load(f'{MODEL_DIR}/Mean-Limb.stl', process=False) # keep raw layout
V_MEAN = mean_limb.vertices.copy()
F_ALL  = mean_limb.faces.copy()

N_SKIN = 57483                          # first 57,483 raw vertices = skin
F_SKIN = F_ALL[: N_SKIN // 3]           # 3 corners per triangle
F_BONE = F_ALL[N_SKIN // 3:] - N_SKIN   # renumber bone faces from 0

print('Raw vertices:', V_MEAN.shape, ' Faces:', F_ALL.shape)
print('Skin faces:', F_SKIN.shape, ' Bone faces:', F_BONE.shape)

Raw vertices: (116730, 3)  Faces: (38910, 3)
Skin faces: (19161, 3)  Bone faces: (19749, 3)


## Merge duplicate skin points the same way for every limb


In [ ]:
_, first_idx, inverse = np.unique(np.round(V_MEAN[:N_SKIN], 9), axis=0,
                                  return_index=True, return_inverse=True)
SKIN_KEEP  = first_idx                  # raw vertex to keep for each unique point
SKIN_FACES = inverse.reshape(-1)[F_SKIN]  # faces renumbered to the merged points

print('Unique skin points:', len(SKIN_KEEP))

Unique skin points: 9650


## Statistical shape model (PCA modes + regression)

32 modes which are the directions the shape can change in.

new limb = avg limb + score1*mode1 + score2\*mode2 + ...


In [ ]:
MODES = np.load(f'{MODEL_DIR}/Components16.npy').astype(np.float64)
with open(f'{MODEL_DIR}/LR.pkl', 'rb') as f: # avoid impossible limbs with linear regression
    LR = pickle.load(f)

SCORE_LO = np.array([-17.04,-10.38,-8.41,-6.42,-4.73,-4/3,-4.01,-2.55,-2.26,-1.97,-1.54,-0.96,
                     -1.46,-0.96,-1.12,-0.95,-0.9,-0.7,-0.66,-0.49,-0.62,-0.53,-0.61,-0.45,
                     -0.37,-0.46,-0.39,-0.32,-0.28,-0.32,-0.22,-0.22])
SCORE_HI = np.array([24.25,16.42,10.04,11.66,5.21,4.38,4.01,2.68,2.5,2.49,2.2,2.48,
                     2.49,1.66,1.65,1.14,0.97,0.83,1.23,0.96,0.73,0.69,0.75,0.54,
                     0.5,0.5,0.46,0.42,0.35,0.32,0.26,0.37])

# scale limbs to real size
TIBIA_MIN, TIBIA_MAX = 313.05, 466.34
print('Modes:', MODES.shape)

Modes: (32, 350190)


## Create virtual patient function

In [ ]:
def make_patient(rng):
    skin_scores = rng.uniform(SCORE_LO, SCORE_HI)
    full_scores = LR.predict(skin_scores[None, :])[0]  # linear regression to full limb scores
    V = V_MEAN + (full_scores @ MODES).reshape(-1, 3)
    tibia_mm = rng.uniform(TIBIA_MIN, TIBIA_MAX)  # add displacement to avg limb
    V = V * tibia_mm  # scale to mm
    return {
        'skin_v': V[:N_SKIN][SKIN_KEEP],  # merged skin points (mm)
        'bone_v': V[N_SKIN:],  # bone points (mm)
        'scores': full_scores,
        'tibia_mm': tibia_mm,
    }

rng = np.random.default_rng(seed=42)
p = make_patient(rng)
print('Skin points:', p['skin_v'].shape, ' Bone points:', p['bone_v'].shape)
print('Tibia length (mm):', round(p['tibia_mm'], 1))
print('Limb size x/y/z (mm):', np.round(np.ptp(p['skin_v'], axis=0), 1))

Skin points: (9650, 3)  Bone points: (59247, 3)
Tibia length (mm): 363.0
Limb size x/y/z (mm): [109.3 151.3 274.7]


## 3D Visualization


In [ ]:
def show_limb(p, color=None, title='Virtual limb', colorbar_title=''):
    sv, bv = p['skin_v'], p['bone_v']
    skin_kw = dict(opacity=0.35, color='tan') if color is None else \  # see-through to see bones
              dict(intensity=color, colorscale='Turbo', opacity=1.0,
                   colorbar=dict(title=colorbar_title))
    fig = go.Figure([
        go.Mesh3d(x=sv[:,0], y=sv[:,1], z=sv[:,2],
                  i=SKIN_FACES[:,0], j=SKIN_FACES[:,1], k=SKIN_FACES[:,2], name='skin', **skin_kw),
        go.Mesh3d(x=bv[:,0], y=bv[:,1], z=bv[:,2],
                  i=F_BONE[:,0], j=F_BONE[:,1], k=F_BONE[:,2], color='ivory', name='bone',
                  opacity=1.0 if color is None else 0.0),
    ])
    fig.update_layout(title=title, scene=dict(aspectmode='data'), height=600)
    fig.show()

show_limb(p)

## Find soft tissue thickness


In [ ]:
# finds tissue thickness to analyze pressure
def tissue_thickness(p):
    dist, _ = cKDTree(p['bone_v']).query(p['skin_v']) # finds nearest point
    return dist

thick = tissue_thickness(p)
print(f'Thickness (mm): min {thick.min():.1f}, median {np.median(thick):.1f}, max {thick.max():.1f}')
show_limb(p, color=thick, title='Soft tissue thickness', colorbar_title='mm')

Thickness (mm): min 0.3, median 27.9, max 87.1


## Close the mesh to measure volume





In [ ]:
# cap mesh to measure volume
def capped_mesh(skin_v):
    m = trimesh.Trimesh(skin_v, SKIN_FACES, process=False)
    sorted_e = m.edges_sorted
    _, inv, counts = np.unique(sorted_e, axis=0, return_inverse=True, return_counts=True)
    boundary = m.edges[counts[inv.reshape(-1)] == 1]
    center = skin_v[np.unique(boundary)].mean(axis=0)
    ci = len(skin_v)
    cap = np.column_stack([boundary[:, 1], boundary[:, 0], np.full(len(boundary), ci)])
    return trimesh.Trimesh(np.vstack([skin_v, center]), np.vstack([SKIN_FACES, cap]), process=False)

# get volume
def volume_ml(skin_v):
    return capped_mesh(skin_v).volume / 1000.0      # mm^3 -> mL

m = capped_mesh(p['skin_v'])
print('Watertight:', m.is_watertight)
print(f'Volume: {volume_ml(p["skin_v"]):.0f} mL')

Watertight: True
Volume: 1931 mL


## Make volume versions (morning, midday, evening)


In [ ]:
VOLUME_CHANGES = [-0.015, -0.0075, 0.0, +0.01, +0.02]    # from literature
BONE_MARGIN_MM = 5.0

def volume_version(p, target_frac, iters=6):
    sv = p['skin_v']
    if target_frac == 0:
        return sv.copy()
    thick = tissue_thickness(p)
    weight = np.clip(thick - BONE_MARGIN_MM, 0, None)
    weight = weight / weight.max()
    normals = trimesh.Trimesh(sv, SKIN_FACES, process=False).vertex_normals
    v0 = volume_ml(sv)

    def moved(s):
        step = s * weight
        step = np.maximum(step, -0.5 * thick)  # never shrink more than halfway to bone
        return sv + step[:, None] * normals

    def error(s):
        return (volume_ml(moved(s)) - v0) / v0 - target_frac

    s0, s1 = 0.0, 5.0 * np.sign(target_frac)  # two starting guesses (mm)
    e0, e1 = error(s0), error(s1)
    for _ in range(iters):
        if abs(e1) < 1e-4 or e1 == e0:
            break
        s0, s1, e0 = s1, s1 - e1 * (s1 - s0) / (e1 - e0), e1
        e1 = error(s1)
    return moved(s1)

v_base = volume_ml(p['skin_v'])
for frac in VOLUME_CHANGES:
    sv_new = volume_version(p, frac)
    got = (volume_ml(sv_new) - v_base) / v_base
    shift = np.linalg.norm(sv_new - p['skin_v'], axis=1)
    print(f'target {frac:+.1%}  got {got:+.2%}  biggest skin shift {shift.max():.1f} mm')

target -1.5%  got -1.50%  biggest skin shift 1.2 mm
target -0.8%  got -0.75%  biggest skin shift 0.6 mm
target +0.0%  got +0.00%  biggest skin shift 0.0 mm
target +1.0%  got +1.00%  biggest skin shift 0.8 mm
target +2.0%  got +2.00%  biggest skin shift 1.6 mm


Analyze where shrinked version skin moved

In [ ]:
shrunk = volume_version(p, VOLUME_CHANGES[0])
show_limb(p, color=np.linalg.norm(shrunk - p['skin_v'], axis=1),
          title=f'How far the skin moved at {VOLUME_CHANGES[0]:+.0%} volume', colorbar_title='mm')

## Generate the whole dataset and split it by patient

Saved as one compressed `.npz` file


In [ ]:
N_PATIENTS = 200
SPLIT = {'train': 0.70, 'val': 0.15, 'test': 0.15}

rng = np.random.default_rng(seed=2026)
skins, bones, thicks, scores, tibias = [], [], [], [], []
MIN_DISTAL_MM = 2.0  # tibia end has to sit this far at the minimum inside the skin tip
TIBIA = slice(29817, 51903)  # tibia's rows in the bone array
rejected = 0
for i in range(N_PATIENTS):
    while True:
        p = make_patient(rng)
        gap = p['bone_v'][TIBIA][:, 2].min() - p['skin_v'][:, 2].min()
        if gap >= MIN_DISTAL_MM:
            break
        rejected += 1   # if bone is through or right at the skin draw again
    versions = [volume_version(p, f) for f in VOLUME_CHANGES]
    skins.append(np.stack(versions))  # (n_versions, n_skin_points, 3)
    bones.append(p['bone_v'])
    thicks.append(tissue_thickness(p))
    scores.append(p['scores']); tibias.append(p['tibia_mm'])
    if (i + 1) % 5 == 0:
        print(f'{i + 1}/{N_PATIENTS} patients done')

# split by patient
order = rng.permutation(N_PATIENTS)
n_tr = int(SPLIT['train'] * N_PATIENTS); n_va = int(SPLIT['val'] * N_PATIENTS)
groups = np.empty(N_PATIENTS, dtype='<U5')
groups[order[:n_tr]] = 'train'
groups[order[n_tr:n_tr + n_va]] = 'val'
groups[order[n_tr + n_va:]] = 'test'

path = f'{OUT_DIR}/virtual_patients_v1.npz'
np.savez_compressed(path,
    skin=np.stack(skins).astype(np.float32), bone=np.stack(bones).astype(np.float32),
    thickness=np.stack(thicks).astype(np.float32), scores=np.stack(scores),
    tibia_mm=np.array(tibias), groups=groups,
    volume_changes=np.array(VOLUME_CHANGES), skin_faces=SKIN_FACES, bone_faces=F_BONE)
print('Saved', path)
print({g: int((groups == g).sum()) for g in SPLIT})
print('Rejected as anatomically impossible:', rejected)

5/200 patients done
10/200 patients done
15/200 patients done
20/200 patients done
25/200 patients done
30/200 patients done
35/200 patients done
40/200 patients done
45/200 patients done
50/200 patients done
55/200 patients done
60/200 patients done
65/200 patients done
70/200 patients done
75/200 patients done
80/200 patients done
85/200 patients done
90/200 patients done
95/200 patients done
100/200 patients done
105/200 patients done
110/200 patients done
115/200 patients done
120/200 patients done
125/200 patients done
130/200 patients done
135/200 patients done
140/200 patients done
145/200 patients done
150/200 patients done
155/200 patients done
160/200 patients done
165/200 patients done
170/200 patients done
175/200 patients done
180/200 patients done
185/200 patients done
190/200 patients done
195/200 patients done
200/200 patients done
Saved /content/drive/MyDrive/all-day-socket/data/virtual_patients_v1.npz
{'train': 140, 'val': 30, 'test': 30}
Rejected as anatomically impo

## Sanity check


In [ ]:
# checks volume changes to ensure good training data
d = np.load(path)
for i in range(3):
    base = volume_ml(d['skin'][i, VOLUME_CHANGES.index(0.0)])
    got = [(volume_ml(d['skin'][i, k]) - base) / base for k in range(len(VOLUME_CHANGES))]
    print(f'patient {i}: volume changes', [f'{g:+.1%}' for g in got])
print('Thinnest tissue across all patients (mm):', d['thickness'].min().round(1))
print('Tibia lengths (mm): min', d['tibia_mm'].min().round(0), 'max', d['tibia_mm'].max().round(0))

patient 0: volume changes ['-1.5%', '-0.8%', '+0.0%', '+1.0%', '+2.0%']
patient 1: volume changes ['-1.5%', '-0.7%', '+0.0%', '+1.0%', '+2.0%']
patient 2: volume changes ['-1.5%', '-0.7%', '+0.0%', '+1.0%', '+2.0%']
Thinnest tissue across all patients (mm): 0.1
Tibia lengths (mm): min 314.0 max 466.0
